## Inputs


## Direct loading preserves selector and scaling conventions

The visible qfunc allocates two asset fields, a five-bit auxiliary selector and
an indicator, then composes exponential loading and reflection helpers.
FlagQuantum builds these declared quantum operations. Host code determines
rounded coefficients, the strike threshold and low-branch rotation from the
finite financial model; these are not inferred compiler choices.

Try zero and one Grover power before altering model data. Check all sixteen
joint selectors, their low-bit ordering and the strike boundary before
comparing price. The iterative estimator's confidence interval accounts for
sampling, not discretization or financial assumptions. Dropped private
registers are retained environment rather than zero workspace.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import allocate
from flagquantum.synth import drop
from flagquantum.synth import phase
from flagquantum.synth import qfunc
from typing import Annotated

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.main import (
    pi,
    prepare,
    reflect,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.overview.workflow import (
    MAX_POWER,
    Model,
    direct,
)


@qfunc
def rainbow(
    k: int,
    probabilities: list[float],
    angles: list[float],
    lower_angle: float,
    indicator: Output[QBit],
) -> None:
    x: Annotated[QNum, 2] = QNum()
    y: Annotated[QNum, 2] = QNum()
    auxiliary: Annotated[QNum, 5] = QNum()
    allocate(x)
    allocate(y)
    allocate(auxiliary)
    allocate(indicator)
    prepare(probabilities, angles, lower_angle, x, y, auxiliary, indicator)
    for iteration in range(k):
        phase(indicator, pi)
        reflect(probabilities, angles, lower_angle, x, y, auxiliary, indicator)
    drop(x)
    drop(y)
    drop(auxiliary)


def build_preview(model: Model, power: int) -> synth.Program:
    if type(power) is not int or not 0 <= power <= MAX_POWER:
        raise ValueError("Use a Grover power in 0..4")
    return synth.synthesize(
        direct,
        classical_inputs=dict(
            k=power,
            probabilities=model.probabilities,
            angles=model.angles,
            lower_angle=model.lower_angle,
        ),
        max_qubits=25,
        max_operations=100_000,
        memory_limit_bytes=512 * 1024**2,
    )


from IPython.display import Image, display

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.overview.main import (
    report,
    save_figure,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.overview.model import (
    ALPHA,
    ASSERTION_ALPHA,
    EPSILON,
    FRACTION_BITS,
    SELECTOR_BITS,
    SOURCE_PRICE,
    STRIKE,
    fresh_model,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.overview.workflow import (
    build_program,
    run_pricing,
)


## Correlated log-return model


In [ ]:
model = fresh_model()
print("Rounded coefficients:", model.coefficients)


## Original estimation settings


In [ ]:
print("Strike, epsilon and alpha:", STRIKE, EPSILON, ALPHA)


## Discretized Gaussian bins


In [ ]:
print("Grid and normalized probabilities:", model.grid, model.probabilities)


## Payoff support


In [ ]:
print("Payoff range:", model.payoff[0], model.payoff[-1])


## Fixed-point precision


In [ ]:
print("Fraction and selector bits:", FRACTION_BITS, SELECTOR_BITS)


## Construct the rounded affine maximum


In [ ]:
print("Rounded offset and all joint selectors:", model.offset, model.selectors)


## Verify the source selector bounds


In [ ]:
assert len(model.selectors) == 16 and all(
    0 <= index < 2**SELECTOR_BITS for index in model.selectors
)


## Direct exponential loading


In [ ]:
print("Exponential rate and five rotations:", model.rate, model.angles)


## Synthesize the complete preparation


In [ ]:
preparation = build_preview(model, 0)
print("Strike threshold and low-branch angle:", model.threshold, model.lower_angle)
print("Preparation resources:", preparation.report["resources"])


## Prepare the count-only estimation profile


In [ ]:
print("Precision and confidence:", EPSILON, 1 - ALPHA)


## Execute the native estimator


In [ ]:
result = run_pricing()
report(result)


## Compute the independent direct-grid expectation


In [ ]:
print("Exact finite expectation:", model.exact_price)


## Inspect the source grid and payoff


In [ ]:
display(Image(filename=str(save_figure(result))))


## Source confidence rule


In [ ]:
assert abs(result.price - SOURCE_PRICE) <= result.source_tolerance
print("Reference confidence level:", 1 - ASSERTION_ALPHA)
